# Security Table Example

This example turns a checked-in live IBKR position artifact into an enriched security table by default.

It also includes an optional live contract-search section, but the default mode is replay-only so the notebook can execute without TWS.

In [ ]:
from pathlib import Path
import sys
from typing import Dict

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


def load_local_account_defaults(root: Path) -> Dict[str, str]:
    candidates = [
        root / "configs" / "portfolio_monitor" / "report_accounts.local.env",
        root / "configs" / "report_accounts.local.env",
    ]
    values: Dict[str, str] = {}
    for path in candidates:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip(chr(34)).strip("'")
        break
    return values


def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

local_account_defaults = load_local_account_defaults(project_root)
artifact_dir = project_root / "data" / "artifacts" / "portfolio_monitor"

SOURCE_MODE = "artifact_replay"  # "artifact_replay" or "live"
RUN_LIVE_LOOKUP = False

ARTIFACT_LIVE_CSV = first_existing_path([
    project_root / "outputs" / "reports" / "live_ibkr_position_report.csv",
    artifact_dir / "live_ibkr_position_report.csv",
])

SYMBOL = "XMME"
SEC_TYPE = "STK"
EXCHANGE = "SMART"
PRIMARY_EXCHANGE = "ARCA"
CURRENCY = "USD"
CONID = None
LOCAL_SYMBOL = None

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 121
IB_ACCOUNT = local_account_defaults.get("DEFAULT_PROD_ACCOUNT_ID", "")

print(f"Project root: {project_root}")
print(f"SOURCE_MODE: {SOURCE_MODE}")
print(f"RUN_LIVE_LOOKUP: {RUN_LIVE_LOOKUP}")
print(f"Artifact CSV: {ARTIFACT_LIVE_CSV}")
print(f"Optional lookup target: symbol={SYMBOL}, sec_type={SEC_TYPE}, exchange={EXCHANGE}, primary_exchange={PRIMARY_EXCHANGE}, currency={CURRENCY}, conid={CONID}")


In [ ]:
import csv
from pprint import pprint

from IPython.display import display
import pandas as pd

from market_helper.portfolio.security_reference import build_security_reference_table


def build_artifact_security_table(csv_path: Path, reference_table):
    rows = list(csv.DictReader(csv_path.open("r", encoding="utf-8", newline="")))
    materialized = []
    for row in rows:
        con_id = str(row.get("con_id") or "").strip()
        symbol = str(row.get("symbol") or "").upper()
        currency = str(row.get("currency") or "").upper()
        security = reference_table.resolve_by_ibkr_conid(con_id) if con_id else None
        if security is None and symbol:
            security = reference_table.resolve_cash_reference(symbol=symbol, currency=currency)
        materialized.append({
            "account": str(row.get("account") or ""),
            "runtime_internal_id": str(row.get("internal_id") or ""),
            "con_id": con_id,
            "symbol": symbol,
            "local_symbol": str(row.get("local_symbol") or ""),
            "exchange": str(row.get("exchange") or ""),
            "currency": currency,
            "quantity": float(row.get("quantity") or 0.0),
            "latest_price": float(row.get("latest_price") or 0.0),
            "market_value": float(row.get("market_value") or 0.0),
            "security_internal_id": security.internal_id if security is not None else "",
            "security_mapping_status": security.mapping_status if security is not None else "unmapped",
            "security_display_ticker": security.display_ticker if security is not None else symbol,
            "security_display_name": security.display_name if security is not None else (str(row.get("local_symbol") or "") or symbol),
            "security_asset_class": security.asset_class if security is not None else "",
            "security_eq_country": security.eq_country if security is not None else "",
            "security_eq_sector": security.eq_sector if security is not None else "",
            "security_fi_tenor": security.fi_tenor if security is not None else "",
            "security_yahoo_symbol": security.yahoo_symbol if security is not None else "",
            "security_lookup_status": security.lookup_status if security is not None else "",
            "security_primary_exchange": security.primary_exchange if security is not None else "",
        })
    return pd.DataFrame(materialized)


reference_table = build_security_reference_table()

if SOURCE_MODE == "live":
    from market_helper.domain.portfolio_monitor import build_live_ibkr_position_security_table

    enriched_table = pd.DataFrame(
        build_live_ibkr_position_security_table(
            host=IB_HOST,
            port=IB_PORT,
            client_id=IB_CLIENT_ID,
            account_id=IB_ACCOUNT or None,
        )
    )
    acquisition_note = "Pulled directly from live TWS / IB Gateway."
else:
    enriched_table = build_artifact_security_table(ARTIFACT_LIVE_CSV, reference_table)
    acquisition_note = "Replayed from the checked-in live IBKR CSV artifact and enriched with generated security reference data."

print(acquisition_note)
print(f"Rows: {len(enriched_table)}")
preview_columns = [
    "account",
    "symbol",
    "local_symbol",
    "market_value",
    "security_display_ticker",
    "security_display_name",
    "security_asset_class",
    "security_eq_country",
    "security_eq_sector",
    "security_fi_tenor",
    "security_yahoo_symbol",
    "security_mapping_status",
]
available_columns = [column for column in preview_columns if column in enriched_table.columns]
display(enriched_table.loc[:, available_columns].reset_index(drop=True))


In [ ]:
if enriched_table.empty or "symbol" not in enriched_table.columns:
    print("No enriched rows were returned.")
    display(enriched_table)
else:
    selected_rows = enriched_table.loc[enriched_table["symbol"].eq(SYMBOL)]
    if selected_rows.empty:
        print(f"No row matched SYMBOL={SYMBOL}; showing the first row instead.")
        display(enriched_table.head(1).T)
    else:
        display(selected_rows.head(1).T)


In [ ]:
if RUN_LIVE_LOOKUP:
    try:
        from market_helper.providers.tws_ib_async import TwsIbAsyncClient

        lookup_client = TwsIbAsyncClient(
            host=IB_HOST,
            port=IB_PORT,
            client_id=IB_CLIENT_ID + 1,
            account=IB_ACCOUNT,
        )
        try:
            lookup_client.connect()
            security_matches = lookup_client.search_securities(
                symbol=SYMBOL,
                sec_type=SEC_TYPE,
                exchange=EXCHANGE,
                primary_exchange=PRIMARY_EXCHANGE,
                currency=CURRENCY,
                conid=CONID,
                local_symbol=LOCAL_SYMBOL,
            )
        finally:
            lookup_client.disconnect()

        print(f"Live IBKR matches: {len(security_matches)}")
        if security_matches:
            display(pd.DataFrame(security_matches).reset_index(drop=True))
        else:
            print("No live matches returned.")
    except Exception as exc:
        print("Live lookup failed cleanly:")
        print(type(exc).__name__, exc)
else:
    print("RUN_LIVE_LOOKUP is False. Set it to True when TWS / IB Gateway is available.")
